In [57]:
import pandas as pd
import numpy as np
from pathlib import Path

In [58]:
PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_IMAGE = PROJECT_ROOT / "data" / "images"
DATA_IMAGE.mkdir(parents=True, exist_ok=True)

print("Processed data folder:", DATA_PROCESSED)

Processed data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\processed


In [59]:
df = pd.read_csv(DATA_PROCESSED / "01_processed.csv")
df = df.sort_values(["user_id", "trajectory_id"])
df

,user_id,timestamp,trajectory_id,ftid,x,y,w,h
0,11,0,ckz3v9nzv00033867jsekqdcl,0,0.275781,0.412500,0.026562,0.037500
1,11,10,ckz3v9nzv00033867jsekqdcl,0,0.275643,0.412992,0.026562,0.037500
2,11,20,ckz3v9nzv00033867jsekqdcl,0,0.275505,0.413483,0.026562,0.037500
3,11,30,ckz3v9nzv00033867jsekqdcl,0,0.275366,0.413975,0.026562,0.037500
4,11,40,ckz3v9nzv00033867jsekqdcl,0,0.275228,0.414467,0.026562,0.037500
...,...,...,...,...,...,...,...,...
58717,82,740,cl5mohjd6000e3b6g97eyecxk,0,0.577263,0.676545,0.031250,0.039583
58718,82,750,cl5mohjd6000e3b6g97eyecxk,0,0.577083,0.675347,0.031250,0.039583
58719,82,760,cl5mohjd6000e3b6g97eyecxk,0,0.576904,0.674150,0.031250,0.039583
58720,82,770,cl5mohjd6000e3b6g97eyecxk,0,0.576724,0.672953,0.031250,0.039583


In [60]:
df = df.sort_values(["trajectory_id", "timestamp"]).reset_index(drop=True)

df['x_translate'] = df.groupby('trajectory_id')['x'].transform(lambda x: x - x.iloc[0])
df['y_translate'] = df.groupby('trajectory_id')['y'].transform(lambda y: y - y.iloc[0])

angles = {}

def pca_angle(x, y):
    X = np.vstack([x, y]).T
    Xc = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    v1 = Vt[0]
    disp = np.array([x[-1], y[-1]])
    if np.dot(v1, disp) < 0:
        v1 = -v1
    return np.arctan2(v1[1], v1[0])

df['x_rotated'] = np.nan
df['y_rotated'] = np.nan

for tid, g in df.groupby('trajectory_id'):
    x, y = g['x_translate'].to_numpy(), g['y_translate'].to_numpy()
    theta = pca_angle(x, y)
    c, s = np.cos(-theta), np.sin(-theta)
    xr = x * c - y * s
    yr = x * s + y * c
    df.loc[g.index, 'x_rotated'] = xr
    df.loc[g.index, 'y_rotated'] = yr
    angles[tid] = theta


In [61]:
df = df.sort_values(["trajectory_id", "timestamp"])

dx = df.groupby("trajectory_id")["x_rotated"].diff()
dy = df.groupby("trajectory_id")["y_rotated"].diff()

df["speed"] = np.hypot(dx, dy)
df["angle_deg"] = np.degrees(np.arctan2(dy, dx))

df.to_csv(DATA_PROCESSED / "02_processed.csv", index=False)
df


,user_id,timestamp,trajectory_id,ftid,x,y,w,h,x_translate,y_translate,x_rotated,y_rotated,speed,angle_deg
0,14,0,ckyw6zzlj001r3867thf0fuy7,0,0.208594,0.825000,0.035937,0.037500,0.000000,0.000000,0.000000,0.000000,NaN,NaN
1,14,10,ckyw6zzlj001r3867thf0fuy7,0,0.166406,0.655833,0.035937,0.037500,-0.042187,-0.169167,0.165227,-0.055652,0.174348,-18.614460
2,14,20,ckyw6zzlj001r3867thf0fuy7,0,0.196094,0.620602,0.035937,0.037500,-0.012500,-0.204398,0.202732,-0.028893,0.046072,35.507444
3,14,30,ckyw6zzlj001r3867thf0fuy7,0,0.230022,0.603869,0.035937,0.037500,0.021429,-0.221131,0.222138,0.003581,0.037830,59.137100
4,14,40,ckyw6zzlj001r3867thf0fuy7,0,0.269531,0.617708,0.035937,0.037500,0.060938,-0.207292,0.211520,0.044074,0.041863,104.693037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58717,54,1420,cl696lj76000r3b6gbse56a1a,0,0.415188,0.141667,0.028125,0.041667,0.005767,0.000000,0.005767,0.000000,0.000230,180.000000
58718,54,1430,cl696lj76000r3b6gbse56a1a,0,0.414959,0.141667,0.028125,0.041667,0.005537,0.000000,0.005537,0.000000,0.000230,180.000000
58719,54,1440,cl696lj76000r3b6gbse56a1a,0,0.414729,0.141667,0.028125,0.041667,0.005307,0.000000,0.005307,0.000000,0.000230,180.000000
58720,54,1450,cl696lj76000r3b6gbse56a1a,0,0.414499,0.141667,0.028125,0.041667,0.005078,0.000000,0.005078,0.000000,0.000230,180.000000


In [ ]:
k = 10

df_reduzido = (
    df[df["timestamp"] % k == 0]
    .reset_index(drop=True)
)
print(f"Antes: {df.shape}\nDepois: {df_reduzido.shape}")